# ActivIQ: end-to-end run (9B zero-shot benchmark + APE + auto-labelling + LoRA 2B)
One notebook, top to bottom. **Runtime > Change runtime type > A100 GPU** (L4/T4 work but are far slower).

Before **Run all**: read `prompts/auto_labelling/baseline.txt` on GitHub (it is printed in step 3), then set `PROMPT_FROZEN = True` in step 1. The run stops early if you have not, so nothing expensive is wasted.

Saved cell outputs show the original A100 run. A fresh run writes to Google Drive (`MyDrive/ActivIQ_runs_pinned`). After a disconnect, reuse that directory and the same project revision to resume.
Never use the test split for prompt selection or training.

In [1]:
# 1. Config + Drive + repo (clones once; never deletes an existing checkout).
import os, subprocess, sys, time
T0 = time.time()
def elapsed(): print(f"[elapsed {(time.time()-T0)/60:.1f} min]")
def run_script(script, *args):
    subprocess.run([sys.executable, f"scripts/{script}.py", *args], check=True)

PROMPT_FROZEN = False   # <-- set True after you have read/frozen prompts/auto_labelling/baseline.txt

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ActivIQ_runs_pinned"
os.makedirs(DRIVE, exist_ok=True)

REPO, BRANCH = "jiakai2002/ActivIQ", "main"
# Private repo? Add GITHUB_TOKEN in Colab Secrets (key icon), then uncomment:
# from google.colab import userdata; os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
tok = os.environ.get("GITHUB_TOKEN")
url = f"https://{tok+'@' if tok else ''}github.com/{REPO}.git"
if not os.path.exists("/content/ActivIQ"):
    !git clone -q --depth 1 -b {BRANCH} {url} /content/ActivIQ
%cd /content/ActivIQ
if not os.path.lexists("reports"):
    os.symlink(DRIVE, "reports")
if not os.path.islink("reports") or os.path.realpath("reports") != DRIVE:
    raise RuntimeError("Existing reports path belongs to another run. Start a fresh Colab runtime.")
!git log --oneline -1
elapsed()

Mounted at /content/drive
/content/ActivIQ
70ca387 (grafted, HEAD -> lora-finetuning, origin/lora-finetuning) Notebook: add 9B zero-shot benchmark to the pipeline notebook
[elapsed 0.7 min]


In [2]:
# 2. Dependencies (keep Colab's torch) + pick settings from the GPU.
# flash-linear-attention = fast kernels for Qwen3.5's linear-attention layers. causal-conv1d is skipped on purpose
# (it compiles from source and often hangs on Colab; the PyTorch fallback for it is cheap).
!pip install -q "transformers>=5.3,<6" "peft>=0.13,<1" "accelerate>=1,<2" safetensors bitsandbytes pandas spacy flash-linear-attention
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type."
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
BF16 = torch.cuda.is_bf16_supported()
# 9B in bf16 needs ~19 GB: use it when the GPU has room (A100). Otherwise 4-bit.
Q9B = "none" if mem >= 30 else "4bit"
# 2B: full precision on bf16 GPUs, 4-bit on a T4.
Q2B = "none" if BF16 else "4bit"
print(torch.__version__, gpu, f"{mem:.0f} GB")
print("9B quantization:", Q9B, "| 2B quantization:", Q2B)
if "T4" in gpu or "L4" in gpu:
    print("WARNING: this GPU is slow for the 9B stages (estimate: many hours). An A100 is much faster.")
elapsed()

2.14.1+cu130 NVIDIA A100-SXM4-40GB 42 GB
9B quantization: none | 2B quantization: none
[elapsed 1.0 min]


In [3]:
# 3. Preflight (no model): checks the human-reviewed seeds and validation labels. Never bypass this.
run_script("auto_label", "check")
print("\n=== prompts/auto_labelling/baseline.txt ===")
print(open("prompts/auto_labelling/baseline.txt").read())

{
  "ready": true,
  "checks": {
    "seeds": {
      "ready": true,
      "reviewed": 10
    },
    "validation": {
      "ready": true,
      "reviewed": 100
    }
  },
  "source_counts": {
    "train": 960,
    "validation": 120,
    "test": 120
  },
  "model_loaded": false
}

=== prompts/auto_labelling/baseline.txt ===
Assign the primary intent and extract only the attributes actually expressed in
this utterance. Follow the fixed task contract and the ten seed examples.
Pay attention to negation, named exercise variants and preparation-specific food
phrases. Canonicalise aliases without broadening specific requests. Keep food
and nutrient entities separate. Do not infer an exercise's muscles as explicitly
stated body-part entities, and do not confuse workout days with session duration.
Use empty lists for absent entities. Check that all eight fields are present and
return only the required intent/entities JSON object.



In [4]:
# 3b. Stop early if the baseline prompt has not been reviewed (fails fast, before any expensive step).
assert PROMPT_FROZEN, "Read the baseline prompt printed above, then set PROMPT_FROZEN = True in step 1 and Run all again."
print("baseline prompt frozen")

baseline prompt frozen


In [5]:
# 4. Download the pinned checkpoints into the local Hugging Face cache (~19 GB + ~5 GB).
import json
from pathlib import Path
from huggingface_hub import snapshot_download
revs = json.loads(Path("ActivIQ_runs/model_revisions.json").read_text(encoding="utf-8"))
QWEN_9B = snapshot_download("Qwen/Qwen3.5-9B", revision=revs["Qwen/Qwen3.5-9B"])
QWEN_2B = snapshot_download("Qwen/Qwen3.5-2B", revision=revs["Qwen/Qwen3.5-2B"])
Path(DRIVE, "model_revisions.json").write_text(json.dumps(revs, indent=2), encoding="utf-8")
print(revs)
elapsed()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

ok models/qwen3.5-9b


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

ok models/qwen3.5-2b
{'Qwen/Qwen3.5-9B': 'c202236235762e1c871ad0ccb60c8ee5ba337b9a', 'Qwen/Qwen3.5-2B': '15852e8c16360a2fea060d615a32b45270f8a8fc'}
[elapsed 2.2 min]


In [6]:
# 4b. Zero-shot 9B benchmark on the 120 test utterances (skipped if results are already on Drive).
# Entity F1 on test is only scored if entities_test.jsonl is human-labelled. If any row is not, --allow-draft is
# needed to run at all and entity F1 stays 'pending_human_annotation' (report this; do not auto-fill labels).
rows = [json.loads(l) for l in open("data/evaluation/entities_test.jsonl")]
statuses = sorted({r.get("label_status", "?") for r in rows})
print("test entity label_status values:", statuses)
DRAFT = [] if statuses == ["human_reviewed"] else ["--allow-draft"]
print("evaluate flag:", DRAFT or "(none)")
if os.path.exists("reports/evaluation/qwen9b_zeroshot/lora_comparison.json"):
    print("zero-shot results already exist, skipping")
else:
    run_script("evaluate_lora", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
               "--skip-baseline", *DRAFT, "--output-dir", "reports/evaluation/qwen9b_zeroshot")
elapsed()

test entity label_status values: ['human_reviewed']
evaluate flag: (none)
Loading weights: 100% 760/760 [00:00<00:00, 5725.06it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
lora: acc=0.8000 macroF1=0.8079 entityF1=0.40764331210191085 median=4.079568929000004
[elapsed 12.1 min]


In [7]:
# 5. APE step 1: the 9B model proposes candidate prompts from the 10 seeds.
run_script("auto_label", "propose", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
           "--candidate-count", "4", "--run-dir", "reports/auto_labelling/propose", "--resume")
elapsed()

Fingerprinting checkpoint file: chat_template.jinja
Fingerprinting checkpoint file: config.json
Fingerprinting checkpoint file: merges.txt
Fingerprinting checkpoint file: model.safetensors-00001-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00002-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00003-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00004-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors.index.json
Fingerprinting checkpoint file: preprocessor_config.json
Fingerprinting checkpoint file: tokenizer.json
Fingerprinting checkpoint file: tokenizer_config.json
Fingerprinting checkpoint file: video_preprocessor_config.json
Fingerprinting checkpoint file: vocab.json
Loading local model weights...
Loading weights: 100% 760/760 [00:04<00:00, 161.40it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is c

In [8]:
# 6. APE step 2: score the baseline + candidates on the 100 validation rows.
run_script("auto_label", "score", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
           "--candidates", "reports/auto_labelling/propose/candidates.jsonl",
           "--run-dir", "reports/auto_labelling/score", "--resume")
elapsed()

Fingerprinting checkpoint file: chat_template.jinja
Fingerprinting checkpoint file: config.json
Fingerprinting checkpoint file: merges.txt
Fingerprinting checkpoint file: model.safetensors-00001-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00002-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00003-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00004-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors.index.json
Fingerprinting checkpoint file: preprocessor_config.json
Fingerprinting checkpoint file: tokenizer.json
Fingerprinting checkpoint file: tokenizer_config.json
Fingerprinting checkpoint file: video_preprocessor_config.json
Fingerprinting checkpoint file: vocab.json
Loading local model weights...
Loading weights: 100% 760/760 [00:04<00:00, 160.50it/s]
Scoring baseline on 100 validation utterances...
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation be

In [9]:
# 7. Which prompt won? (validation selection scores, NOT held-out performance)
import pandas as pd
sel = json.load(open("reports/auto_labelling/score/selection.json"))
print("Selected prompt:", sel["selected_prompt"]["id"], "| entity target met:", sel["validation_entity_target_met"])
display(pd.DataFrame([{"prompt": r["id"], "entity_micro_f1": r["metrics"]["micro"]["f1"],
    "valid_output_rate": r["metrics"]["valid_output_rate"], "intent_agreement": r["metrics"]["intent_agreement"]}
    for r in sel["ranking"]]))
print("Targets: entity F1 >= 0.80, intent agreement >= 0.85 (proposal section 5)")

Selected prompt: ape-002 | entity target met: False


,prompt,entity_micro_f1,valid_output_rate,intent_agreement
0,ape-002,0.700000,1.0,0.92
1,ape-003,0.670732,1.0,0.93
2,baseline,0.658683,1.0,0.92
3,ape-004,0.646707,1.0,0.92
4,ape-001,0.642857,1.0,0.92


Targets: entity F1 >= 0.80, intent agreement >= 0.85 (proposal section 5)


In [10]:
# 8. Label the 960 training utterances with the selected prompt (the long step; safe to resume).
run_script("auto_label", "label", "--model-path", QWEN_9B, "--device", "cuda", "--quantization", Q9B,
           "--selection", "reports/auto_labelling/score/selection.json",
           "--run-dir", "reports/auto_labelling/train_labels", "--resume")
elapsed()

Fingerprinting checkpoint file: chat_template.jinja
Fingerprinting checkpoint file: config.json
Fingerprinting checkpoint file: merges.txt
Fingerprinting checkpoint file: model.safetensors-00001-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00002-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00003-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors-00004-of-00004.safetensors
Fingerprinting checkpoint file: model.safetensors.index.json
Fingerprinting checkpoint file: preprocessor_config.json
Fingerprinting checkpoint file: tokenizer.json
Fingerprinting checkpoint file: tokenizer_config.json
Fingerprinting checkpoint file: video_preprocessor_config.json
Fingerprinting checkpoint file: vocab.json
Loading local model weights...
Loading weights: 100% 760/760 [00:04<00:00, 161.47it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is c

In [11]:
# 9. Label quality: how often does the 9B intent match the dataset's source intent?
rows = [json.loads(l) for l in open("reports/auto_labelling/train_labels/machine_labels.jsonl")]
rp = "reports/auto_labelling/train_labels/rejected.jsonl"
rej = sum(1 for _ in open(rp)) if os.path.exists(rp) else 0
agree = sum(r["intent"] == r["source_intent"] for r in rows) / len(rows)
print(f"accepted {len(rows)} | rejected {rej} | intent agreement {agree:.3f} (target >= 0.85)")

accepted 960 | rejected 0 | intent agreement 0.904 (target >= 0.85)


In [17]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [18]:
# 10. LoRA fine-tune Qwen3.5-2B on the machine labels (skipped if the adapter already exists on Drive).
if os.path.exists("reports/lora/adapter_v1/training_report.json"):
    if not os.path.isfile("reports/lora/adapter_v1/adapter_model.safetensors"):
        raise FileNotFoundError("The training report exists, but the adapter weights are missing. Restore them or use a new run directory.")
    print("adapter_v1 already trained, skipping")
else:
    run_script("train_lora", "--model-path", QWEN_2B, "--device", "cuda", "--quantization", Q2B,
               "--labels", "reports/auto_labelling/train_labels/machine_labels.jsonl",
               "--output-dir", "reports/lora/adapter_v1")
elapsed()

Loading weights: 100% 617/617 [00:00<00:00, 5303.88it/s]
trainable params: 23,340,032 || all params: 2,236,581,696 || trainable%: 1.0436
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
epoch 1 step 10/360 loss 0.1238
epoch 1 step 20/360 loss 0.0968
epoch 1 step 30/360 loss 0.0659
epoch 1 step 40/360 loss 0.0791
epoch 1 step 50/360 loss 0.0265
epoch 1 step 60/360 loss 0.1829
epoch 1 step 70/360 loss 0.0116
epoch 1 step 80/360 loss 0.0188
epoch 1 step 90/360 loss 0.0375
[W1109 11:58:21.010747901 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 1509949440 bytes (free: 550961152, total: 42405855232).
epoch 1 step 100/360 loss 0.1905
epoch 1 step 110/360 loss 0.0320
epoch 1 step 120/360 loss 0.0105
epoch 1 mean loss 0.0634
epoch 2 step 130/360 loss 0.0085
epoch 2 step 140/36

### DistilBERT baseline (optional but recommended)
Zip your local `models/intent_classifier/` (~255 MB), upload it to `MyDrive/ActivIQ_runs/intent_classifier.zip`, then the next cell restores it so both systems run on the same machine. Without it the comparison skips DistilBERT and you must report its numbers from your laptop (different hardware for latency).

In [19]:
# 11. Restore DistilBERT if you uploaded it.
z = f"{DRIVE}/intent_classifier.zip"
HAVE_BASELINE = os.path.exists(z)
if HAVE_BASELINE:
    !mkdir -p models && unzip -qo {z} -d models/
    !ls models/intent_classifier | head
print("baseline available:", HAVE_BASELINE)

baseline available: False


In [20]:
# 11b. No DistilBERT zip? Retrain it with the repo's own script (same recipe, new weights).
!if [ ! -f models/intent_classifier/config.json ]; then mkdir -p .tmp && sed 's/^\( *\)tokenizer=tokenizer,$/\1processing_class=tokenizer,/' scripts/train_intent.py > .tmp/train_intent_colab.py && python .tmp/train_intent_colab.py; fi
HAVE_BASELINE = os.path.exists("models/intent_classifier/config.json"); print("baseline available:", HAVE_BASELINE)

baseline available: True


In [21]:
# 12. Evaluate LoRA 2B (and DistilBERT + spaCy if available) on the 120 test utterances.
skip = [] if HAVE_BASELINE else ["--skip-baseline"]
run_script("evaluate_lora", "--model-path", QWEN_2B, "--adapter", "reports/lora/adapter_v1",
           "--device", "cuda", "--quantization", Q2B, *skip, *DRAFT,
           "--output-dir", "reports/evaluation/lora_2b")
elapsed()

Loading weights: 100% 104/104 [00:00<00:00, 5525.11it/s]
Loading weights: 100% 617/617 [00:00<00:00, 5708.94it/s]
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
distilbert_spacy: acc=0.9167 macroF1=0.9160 entityF1=0.45517241379310347 median=0.01783254899964959
lora: acc=0.9167 macroF1=0.9186 entityF1=0.5714285714285714 median=2.905283258500276
[elapsed 116.6 min]


In [22]:
# 13. Final comparison table.
def row(name, r):
    e = r["entities"]; f1 = e["micro"]["f1"] if "micro" in e else e.get("status")
    return {"system": name, "intent_acc": r["intent"]["accuracy"], "intent_macro_f1": r["intent"]["macro_f1"],
            "entity_micro_f1": f1, "median_latency_s": r["latency_seconds"].get("median_seconds"),
            "invalid_outputs": r.get("invalid_outputs", 0)}
rep = json.load(open("reports/evaluation/lora_2b/lora_comparison.json"))
table = []
if "distilbert_spacy" in rep: table.append(row("DistilBERT + spaCy", rep["distilbert_spacy"]))
table.append(row("Qwen3.5-2B + LoRA (machine labels)", rep["lora"]))
zs = "reports/evaluation/qwen9b_zeroshot/lora_comparison.json"
if os.path.exists(zs): table.append(row("Qwen3.5-9B zero-shot", json.load(open(zs))["lora"]))
pd.set_option("display.float_format", "{:.3f}".format)
display(pd.DataFrame(table))
print("Latency: GPU for Qwen, CPU/GPU per your runtime; state the hardware in the report.")
print("Report caveats: seeds/validation labels were AI-drafted then human-reviewed; validation scores are selection scores, not held-out;",
      "test entity F1 is pending unless entities_test.jsonl is human-labelled; model revisions are in model_revisions.json on Drive.")
elapsed()

,system,intent_acc,intent_macro_f1,entity_micro_f1,median_latency_s,invalid_outputs
0,DistilBERT + spaCy,0.917,0.916,0.455,0.018,0
1,Qwen3.5-2B + LoRA (machine labels),0.917,0.919,0.571,2.905,0
2,Qwen3.5-9B zero-shot,0.800,0.808,0.408,4.080,5


Latency: GPU for Qwen, CPU/GPU per your runtime; state the hardware in the report.
Report caveats: seeds/validation labels were AI-drafted then human-reviewed; validation scores are selection scores, not held-out; test entity F1 is pending unless entities_test.jsonl is human-labelled; model revisions are in model_revisions.json on Drive.
[elapsed 117.1 min]


In [23]:
# 14. Everything is already on Drive under ActivIQ_runs/. Also download a zip.
!zip -qr results.zip reports/auto_labelling/score reports/auto_labelling/train_labels/summary.md reports/lora/adapter_v1/training_report.json reports/evaluation {DRIVE}/model_revisions.json
from google.colab import files; files.download("results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>